<a href="https://colab.research.google.com/github/mohamedmahmoud8801/llm-notebooks/blob/main/makeing_encoder_and_decoder_for_converting_tokenization_to_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:


import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)


In [54]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of character:", len(raw_text))
print(raw_text[:99])


Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [55]:
import re
text = "hello,world.this,is a test."

result = re.split(r'(\s)',text)
print(result)

['hello,world.this,is', ' ', 'a', ' ', 'test.']


In [56]:
result = re.split(r'([,.]|\s)',text)
print(result)

['hello', ',', 'world', '.', 'this', ',', 'is', ' ', 'a', ' ', 'test', '.', '']


In [57]:
result = [item for item in result if item.strip()]
print(result)

['hello', ',', 'world', '.', 'this', ',', 'is', 'a', 'test', '.']


In [58]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [59]:
print(len(preprocessed))

4690


In [60]:
all_words= sorted(set(preprocessed))

In [61]:
voc_size = len(all_words)
print(voc_size)

1130


In [62]:
vocab = {token:integer for integer, token in enumerate(all_words)}

In [63]:
for i,item in enumerate(vocab.items()):
  print(item)
  if i >=50:
    break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [64]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text




In [65]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)



[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [66]:
tokenizer.decode(ids)




'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [67]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [68]:
len(vocab.items())

1132

In [69]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)



('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [70]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text



In [71]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)



Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [72]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [73]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

In [74]:
import importlib
import tiktoken
print("tiktoken version :",importlib.metadata.version("tiktoken"))

tiktoken version : 0.13.0


In [75]:
tokenizer = tiktoken.get_encoding("gpt2")

In [76]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [77]:
strings = tokenizer.decode(integers)

In [52]:
strings

'Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.'

In [104]:
integers = tokenizer.encode("AKwirw ier")
print(integers)
strings = tokenizer.decode(integers)
print(strings)

[10206, 86, 343, 86, 220, 959]
AKwirw ier


In [107]:
import tiktoken
encodings = {
    "gpt2":tiktoken.get_encoding("gpt2"),
    "gpt3":tiktoken.get_encoding("p50k_base"),
    "gpt4":tiktoken.get_encoding("cl100k_base"),
    }

vocab_sizes= {model:encoding.n_vocab for model,encoding in encodings.items()}
for model, size in vocab_sizes.items():
  print(f"the vocublary size for {model.upper()} is:{size}")

the vocublary size for GPT2 is:50257
the vocublary size for GPT3 is:50281
the vocublary size for GPT4 is:100277


In [78]:
w  ith open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
enc_text= tokenizer.encode(raw_text)
print(len(enc_text))


5145


In [79]:
enc_sample = enc_text[50:]

In [80]:
context_size= 4
x=enc_sample[:context_size]
y=enc_sample[1:context_size+1]
print(f"x:{x}")
print(f"y:      {y}")


x:[290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [81]:
for i in range(1,context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]

  print(context,"---->",desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [82]:
for i in range(1,context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]

  print(tokenizer.decode(context),"---->",tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [83]:
import torch

In [84]:
from torch.utils.data import Dataset, DataLoader

In [85]:
class GPTDatasetV1(Dataset):
  def __init__(self,txt,tokenizer,max_length,stride):
    self.input_ids=[]
    self.target_ids=[]



    tokens_id = tokenizer.encode(txt,allowed_special={"<|endoftext|>"})
    print("Number of tokens:", len(tokens_id))
    print("max_length:", max_length)
    print("stride:", stride)

    for i in range(0,len(tokens_id)-max_length,stride):
      input_chunk = tokens_id[i:i+max_length]
      target_chunk = tokens_id[i+1:i+max_length+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))
  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx) :
    return self.input_ids[idx],self.target_ids[idx]



In [86]:
def create_dataloader_v1(txt,batch_size=4,max_length=256
                         ,stride=128,shuffle=True,drop_last=True,
                         num_workers=0):

  tokenizer = tiktoken.get_encoding("gpt2")

  dataset = GPTDatasetV1(txt,tokenizer,max_length,stride)


  dataloader = DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle=shuffle,
      drop_last=drop_last,
      num_workers=num_workers,
  )

  return dataloader

In [87]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()



In [88]:
import torch

dataloader= create_dataloader_v1(
    raw_text,batch_size=1,max_length=4,stride=1,shuffle=False

)

data_iter = iter(dataloader)
first_batch = next(data_iter)

print(first_batch)

Number of tokens: 5145
max_length: 4
stride: 1
[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [89]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [90]:
import torch
dataloader= create_dataloader_v1(
    raw_text,batch_size=8,max_length=4,stride=4,shuffle=False

)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("inputs\n",inputs)
print("input\n",targets)

Number of tokens: 5145
max_length: 4
stride: 4
inputs
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
input
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [91]:
print(len(dataloader))

160


In [92]:
input_ids = torch.tensor([2,3,5,1])

In [93]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [94]:
print(embedding_layer(torch.tensor(3)))

tensor([-0.4015,  0.9666, -1.1481], grad_fn=<EmbeddingBackward0>)


In [95]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [96]:
vocab_size=50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)


In [97]:
max_length = 6
dataloader = create_dataloader_v1(
    raw_text,
    batch_size =8 ,
    max_length=max_length,
    stride=max_length,
    shuffle=False,

)
data_iter = iter(dataloader)

inputs,target = next(data_iter)



Number of tokens: 5145
max_length: 6
stride: 6


In [98]:
print("Token IDs:\n",inputs)
print("\nInputs shape:\n",inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464,  1807,  3619],
        [  402,   271, 10899,  2138,   257,  7026],
        [15632,   438,  2016,   257,   922,  5891],
        [ 1576,   438,   568,   340,   373,   645],
        [ 1049,  5975,   284,   502,   284,  3285],
        [  326,    11,   287,   262,  6001,   286],
        [  465, 13476,    11,   339,   550,  5710],
        [  465, 12036,    11,  6405,   257,  5527]])

Inputs shape:
 torch.Size([8, 6])


In [99]:
print("target IDs:",target)

target IDs: tensor([[  367,  2885,  1464,  1807,  3619,   402],
        [  271, 10899,  2138,   257,  7026, 15632],
        [  438,  2016,   257,   922,  5891,  1576],
        [  438,   568,   340,   373,   645,  1049],
        [ 5975,   284,   502,   284,  3285,   326],
        [   11,   287,   262,  6001,   286,   465],
        [13476,    11,   339,   550,  5710,   465],
        [12036,    11,  6405,   257,  5527, 27075]])


In [100]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 6, 256])


In [101]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length,output_dim)

In [102]:
pos_embedding = pos_embedding_layer(torch.arange(max_length))
print(pos_embedding.shape)

torch.Size([6, 256])
